In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import random
import time
from openai import OpenAI
from litellm import completion
from IPython.display import display
from data_prep.parser import scrub, parse
from data_prep.items import Item
from collections import Counter
from groq import Groq
import json
from data_prep.batch import Batch
from data_prep.items import Item
from data_prep import RAGEvaluator

load_dotenv(override=True)

True

In [6]:
groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    print("GROQ_API_KEY is set.")
else:
    print("GROQ_API_KEY is not set.")

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if openrouter_api_key:
    print("OPENROUTER_API_KEY is set.")
else:
    print("OPENROUTER_API_KEY is not set.")

hf_token = os.environ['HF_TOKEN']
if hf_token:
    print("HuggingFace token found.")
else:
    print("No HuggingFace token found.")

login(hf_token, add_to_git_credential=True)

#------------------------------

openrouter_url = "https://openrouter.ai/api/v1"


GROQ_API_KEY is set.
OPENROUTER_API_KEY is set.
HuggingFace token found.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
username = "leearum95"
dataset = f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 7,099 items
category='Sporting Goods' full='SPORTS,LACROSSE RBDR Lacrosse Ball Rebounder, 80" x 3 ft, 1.25" dia Frame With foldable, sturdy construction, quickly and easily set up this bounce back target as a ball return anywhere, Great for backyard or field practice, made with tough mesh netting to withstand hard shots and daily practice. From kids to professionals, our products are designed to help every beginner, intermediate and advanced player refine and expand his or her lacrosse techniques and experience. Used as a tool by players for practice, shooter drills, and game play, each portable target serves as a goalie to field your passes and shots. Bright orange nylon net is the perfect addition to any sports set. Our Price Buys you: 1' summary=None prompt=None id=None


In [8]:
for index, item in enumerate(items):
    item.id = index

Example of preprocessing item's description

In [11]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Summary: 1 sentence description of the product, including key features, purpose use cases. Be concise and informative.
"""

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="gpt-4.1-nano")

print(response.choices[0].message.content)
print()
print(items[0].full)
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

# Process summary in batch mode on OpenAI

In [9]:
Batch.create(items)



Created 8 batches


In [10]:
Batch.run()

  0%|          | 0/8 [00:00<?, ?it/s]

Submitted 8 batches


In [12]:
Batch.fetch()

  0%|          | 0/8 [00:00<?, ?it/s]

Finished 8 of 8 batches


In [ ]:
#checking for items without summaries
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
#wipe unnecessary fields to save space
for item in items:
    item.full = None
    item.id = None

In [ ]:
username = "leearum95"
full = f"{username}/items_full"
# full = f"{username}/items_full_multilang"



train = items[:25000]
val = items[25000:26000]
test = items[26000:]

# print(len(train), len(val), len(test))

Item.push_to_hub(full, train, val, test)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [4]:

# running_statuses = {"validating", "in_progress", "finalizing", "cancelling"}

client = OpenAI()

for file in client.files.list():
    client.files.delete(file.id)
    print("deleted", file.id)

# for batch in client.batches.list(limit=100):
#     if batch.status in running_statuses:
#         print(batch.id, batch.status, batch.endpoint)


# batch_id = "batch_69f24e260f24819086239fbb36d1196a"   # replace with your actual batch id

# batch = client.batches.cancel(batch_id)

# print(batch.id)
# print(batch.status)

deleted file-H65ZZKgKmJsC9tukV4tUDY
deleted file-1S2FEACyavQPNfD7UizqXW
deleted file-14oRjULyKz44b2VCkP95NT
deleted file-ChLJqE8gs3fjPVawEe3zw8
deleted file-5S5qU5V1pWRsZDPvG27iWP
deleted file-ANQdtWvabghi6Sre2ovjHg
deleted file-NGcegBYEiuAXz17QXGk4FR
deleted file-XAKDRPNW826HAVhGJoDkCc
deleted file-1RNVHanuE55YZC5ruKChpm
deleted file-GwPkoy6fn9kxZJu2CYXPW5
deleted file-DJBMBBngMLvcaekAz8i975
deleted file-61MVX63omNqb7n3XbqF9oC
deleted file-3DVaD4Ty4ypZfaceiw1drX
deleted file-1czxR6CgFqmVHGWHikx1dQ
deleted file-9gjWm73WDLMabQ4ktn7x3B
deleted file-TMfJyG8bFoPXovpVhVz3Hx


In [3]:
client = OpenAI()
running_statuses = {"validating", "in_progress", "finalizing", "cancelling"}
for batch in client.batches.list(limit=100):
    if batch.status in running_statuses:
        print(batch.id, batch.status, batch.endpoint)



In [ ]:
# from datetime import datetime
# from zoneinfo import ZoneInfo
# tz = ZoneInfo("America/New_York")

# start = datetime(2026, 4, 29, 15, 4, tzinfo=tz)  # 1:30 PM
# end   = datetime(2026, 4, 29, 17, 45, tzinfo=tz)  # 2:45 PM

# start_ts = int(start.timestamp())
# end_ts = int(end.timestamp())

# for batch in client.batches.list(limit=100):
#     if batch.status == "failed" and batch.failed_at:
#         if start_ts <= batch.failed_at <= end_ts:
#             failed_time = datetime.fromtimestamp(batch.failed_at, tz)
#             created_time = datetime.fromtimestamp(batch.created_at, tz)

#             print("batch id:", batch.id)
#             print("failed at:", failed_time.strftime("%Y-%m-%d %I:%M %p"))
#             print("created at:", created_time.strftime("%Y-%m-%d %I:%M %p"))
#             print("endpoint:", batch.endpoint)
#             print()